# Long-Term Agentic Memory with LangGraph (using Qwen2.5 locally)

An email assistant that gets better the more you correct it — built with [LangGraph](https://langchain-ai.github.io/langgraph/) and [LangMem](https://langchain-ai.github.io/langmem/), running entirely on a **local** Qwen2.5-7B via Ollama.

**Setup**: `uv sync --group langgraph-memory` — see the [README](../../../README.md#setup)
**LLM**: Qwen2.5-7B via Ollama · **Embeddings**: `sentence-transformers/all-MiniLM-L6-v2`, in-process

Pull the model before running:
```bash
ollama pull qwen2.5:7b
```

**Covered topics**:
1. Three Kinds of Memory
2. Semantic Memory
3. Episodic Memory
4. The Triage Router
5. The Email Agent
6. Procedural Memory
7. Watching Behaviour Change

---

### The idea in one paragraph

"Give the agent memory" is three separate mechanisms wearing one name, and conflating them is why memory features disappoint:

| Kind | Holds | Written by | Read when |
|---|---|---|---|
| **Semantic** | facts — who Alice is, what she owns | the agent, via tools | it needs to look something up |
| **Episodic** | past decisions, replayed as few-shot examples | your corrections | it classifies something similar |
| **Procedural** | the instructions it follows | an optimizer, from your feedback | every single turn |

Semantic memory makes the agent *informed*. Episodic memory makes it *consistent*. Only procedural memory makes it **change its behaviour** — and that is the one this notebook builds toward.

All three live in one LangGraph `InMemoryStore`, namespaced per user, so swapping in a persistent store never touches the agent code.

> **Why not Ollama embeddings?** The semantic store needs an embedding endpoint, and Ollama only serves one when started with `--embeddings`. Loading a small sentence-transformers model in-process keeps this runnable on a default install.

In [ ]:
# Dependencies come from the repo's uv environment. Run once in a terminal:
#     uv sync --group langgraph-memory
#
# Then `uv run jupyter lab`, or point your IDE's kernel at .venv/bin/python.
# Syncing a new group while this kernel is running? Restart the kernel after.

## Setup

The LLM, the embedding model, and the store every memory type shares.

In [ ]:
import json

from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command
from langmem import (
    create_manage_memory_tool,
    create_multi_prompt_optimizer,
    create_search_memory_tool,
)
from pydantic import BaseModel, Field
from typing import Annotated, Literal, TypedDict

MODEL = "qwen2.5:7b"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIMS = 384
USER_ID = "john"

llm = ChatOllama(model=MODEL, temperature=0)
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# The index is what makes store.search(..., query=...) a semantic lookup
# rather than a plain listing.
store = InMemoryStore(index={"embed": embeddings, "dims": EMBED_DIMS})

In [ ]:
profile = {
    "name": "John",
    "full_name": "John Doe",
    "user_profile_background": "Senior software engineer leading a team of 5 developers",
}

# Seed values for procedural memory. After the first run these live in the
# store, and the agent — not this cell — decides what they say.
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage John's tasks efficiently.",
}

SAMPLE_EMAIL = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": (
        "Hi John,\n\n"
        "I was reviewing the API documentation for the new authentication service "
        "and noticed a few endpoints seem to be missing from the specs. Could you "
        "clarify whether that was intentional?\n\n"
        "Specifically: /auth/refresh and /auth/validate.\n\nThanks!\nAlice"
    ),
}

Two small helpers. `clean_prompt` is not incidental — see section 6 for why a local model needs it.

In [ ]:
def clean_prompt(text: str) -> str:
    """Strip scaffolding a small model sometimes echoes into its answer."""
    for tag in ("<current_prompt>", "</current_prompt>"):
        text = text.replace(tag, "")
    return text.strip()


def get_prompt(key: str, default: str) -> str:
    """Read a prompt from procedural memory, seeding it on first access."""
    result = store.get((USER_ID,), key)
    if result is None:
        store.put((USER_ID,), key, {"prompt": default})
        return default
    return result.value["prompt"]

## 1. Three Kinds of Memory

Procedural memory is seeded from the dict above the first time it is read, and lives in the store from then on. Note the namespaces: everything is keyed by user, so one agent definition serves many people without leaking memory between them.

In [ ]:
for key, default in [
    ("triage_ignore", prompt_instructions["triage_rules"]["ignore"]),
    ("triage_notify", prompt_instructions["triage_rules"]["notify"]),
    ("triage_respond", prompt_instructions["triage_rules"]["respond"]),
    ("agent_instructions", prompt_instructions["agent_instructions"]),
]:
    get_prompt(key, default)

print(f"Namespace ({USER_ID!r},) now holds the agent's procedural memory:")
for item in store.search((USER_ID,)):
    print(f"  {item.key:20} {item.value['prompt'][:60]}...")

## 2. Semantic Memory

Facts the agent looks up. LangMem supplies two ready-made tools, and the `{langgraph_user_id}` placeholder in the namespace is filled from config at call time — that is what keeps one agent definition multi-tenant.

In [ ]:
manage_memory_tool = create_manage_memory_tool(
    namespace=("email_assistant", "{langgraph_user_id}", "collection")
)
search_memory_tool = create_search_memory_tool(
    namespace=("email_assistant", "{langgraph_user_id}", "collection")
)
print("tools:", manage_memory_tool.name, "|", search_memory_tool.name)

In [ ]:
namespace = ("email_assistant", USER_ID, "collection")
store.put(namespace, "alice", {"content": "Alice Smith owns the authentication service and prefers email over meetings."})
store.put(namespace, "standup", {"content": "The team standup is 09:15 every weekday and John chairs it."})

for query in ["who looks after auth?", "when do we meet?"]:
    hits = store.search(namespace, query=query, limit=1)
    print(f"search({query!r}) -> {hits[0].value['content']}")

## 3. Episodic Memory

Past triage decisions, replayed as few-shot examples. Retrieval is semantic, so the examples that reach the prompt are the ones resembling the email being classified — not an arbitrary sample. This is how a correction you made last week influences a decision today without anyone editing a prompt.

In [ ]:
EXAMPLE_TEMPLATE = """Email Subject: {subject}
Email From: {from_email}
Email To: {to_email}
Email Content:
```
{content}
```
> Triage Result: {result}"""


def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(
            EXAMPLE_TEMPLATE.format(
                subject=eg.value["email"]["subject"],
                to_email=eg.value["email"]["to"],
                from_email=eg.value["email"]["author"],
                content=eg.value["email"]["email_thread"][:400],
                result=eg.value["label"],
            )
        )
    return "\n\n------------\n\n".join(strs)

In [ ]:
ns = ("email_assistant", USER_ID, "examples")
store.put(ns, "ex-spam", {
    "email": {
        "author": "Marketing <promo@vendor.com>",
        "to": "John Doe <john.doe@company.com>",
        "subject": "50% off developer tools this week",
        "email_thread": "Limited time offer on our IDE bundle. Click to claim.",
    },
    "label": "ignore",
})
store.put(ns, "ex-bug", {
    "email": {
        "author": "Bob <bob@company.com>",
        "to": "John Doe <john.doe@company.com>",
        "subject": "Production login failures",
        "email_thread": "Auth service is returning 500s for ~5% of logins since the deploy.",
    },
    "label": "respond",
})

hits = store.search(ns, query=str({"email": SAMPLE_EMAIL}), limit=2)
for hit in hits:
    print(f"[{hit.value['label']:7}] {hit.value['email']['subject']}")
print()
print(format_few_shot_examples(hits)[:300], "...")

## 4. The Triage Router

The first node of the graph. It reads its **rules** from procedural memory and its **examples** from episodic memory, then uses structured output to force exactly one of three labels — so the routing decision is a value your code can branch on, not prose to parse.

> The original course notebook imports `triage_user_prompt` from a `prompts` module that ships with the course workspace. It is inlined below so the tutorial stands alone.

In [ ]:
TRIAGE_SYSTEM_PROMPT = """
< Role >
You are {full_name}'s executive assistant.
</ Role >

< Background >
{user_profile_background}.
</ Background >

< Instructions >
{name} gets lots of emails. Categorise each email into one of three categories:
1. IGNORE  - not worth responding to or tracking
2. NOTIFY  - important information, but no response needed
3. RESPOND - needs a direct response from {name}
</ Instructions >

< Rules >
Emails that are not worth responding to:
{triage_no}

Emails {name} should know about but need no response:
{triage_notify}

Emails worth responding to:
{triage_email}
</ Rules >

< Few shot examples >
Follow these examples more than any instruction above.

{examples}
</ Few shot examples >
"""

TRIAGE_USER_PROMPT = """
Please determine how to handle the below email thread:

From: {author}
To: {to}
Subject: {subject}
{email_thread}"""

In [ ]:
class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(description="Step-by-step reasoning behind the classification.")
    classification: Literal["ignore", "respond", "notify"] = Field(
        description=(
            "The classification of an email: 'ignore' for irrelevant emails, "
            "'notify' for important information that doesn't need a response, "
            "'respond' for emails that need a reply"
        )
    )


llm_router = llm.with_structured_output(Router)


class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]


def build_triage_prompt(email):
    return TRIAGE_SYSTEM_PROMPT.format(
        **profile,
        triage_no=get_prompt("triage_ignore", prompt_instructions["triage_rules"]["ignore"]),
        triage_notify=get_prompt("triage_notify", prompt_instructions["triage_rules"]["notify"]),
        triage_email=get_prompt("triage_respond", prompt_instructions["triage_rules"]["respond"]),
        examples=format_few_shot_examples(
            store.search(("email_assistant", USER_ID, "examples"), query=str({"email": email}))
        ),
    )

In [ ]:
for label, email in [
    ("direct question", SAMPLE_EMAIL),
    ("marketing blast", {
        "author": "Deals <deals@vendor.com>",
        "to": "John Doe <john.doe@company.com>",
        "subject": "Your weekly newsletter",
        "email_thread": "Top 10 productivity hacks, plus 30% off our annual plan.",
    }),
]:
    result = llm_router.invoke([
        {"role": "system", "content": build_triage_prompt(email)},
        {"role": "user", "content": TRIAGE_USER_PROMPT.format(
            author=email["author"], to=email["to"],
            subject=email["subject"], email_thread=email["email_thread"])},
    ])
    print(f"{label:18} -> {result.classification}")

Wrapped as a graph node, the router returns a `Command` that both routes and updates state:

In [ ]:
def triage_router(state: State, config, store) -> Command[Literal["response_agent", "__end__"]]:
    email_input = state["email_input"]
    user_prompt = TRIAGE_USER_PROMPT.format(
        author=email_input["author"], to=email_input["to"],
        subject=email_input["subject"], email_thread=email_input["email_thread"])

    result = llm_router.invoke([
        {"role": "system", "content": build_triage_prompt(email_input)},
        {"role": "user", "content": user_prompt},
    ])
    print(f"  triage -> {result.classification.upper()} ({result.reasoning[:70]}...)")

    if result.classification == "respond":
        return Command(
            goto="response_agent",
            update={"messages": [{"role": "user", "content": f"Respond to this email:\n\n{user_prompt}"}]},
        )
    return Command(goto=END)

## 5. The Email Agent

Three business tools plus the two LangMem memory tools, wired into a ReAct agent. The important detail is `create_prompt`: it reads the system prompt **from the store on every invocation** rather than closing over a constant. That single choice is what lets section 6 change the agent's behaviour without rebuilding the graph.

In [ ]:
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    return f"Email sent to {to} with subject '{subject}'"


@tool
def schedule_meeting(attendees: list[str], subject: str, duration_minutes: int, preferred_day: str) -> str:
    """Schedule a calendar meeting."""
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"


@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

In [ ]:
AGENT_SYSTEM_PROMPT = """
< Role >
You are {full_name}'s executive assistant.
</ Role >

< Tools >
1. write_email(to, subject, content)
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day)
3. check_calendar_availability(day)
4. manage_memory - store anything worth remembering about contacts or decisions
5. search_memory - look up what you stored earlier
</ Tools >

< Instructions >
{instructions}
</ Instructions >
"""


def create_prompt(state, config, store):
    """Rebuild the system prompt from procedural memory on every invocation."""
    user_id = config["configurable"]["langgraph_user_id"]
    result = store.get((user_id,), "agent_instructions")
    instructions = result.value["prompt"] if result else prompt_instructions["agent_instructions"]
    return [{
        "role": "system",
        "content": AGENT_SYSTEM_PROMPT.format(instructions=instructions, **profile),
    }] + state["messages"]

In [ ]:
response_agent = create_react_agent(
    llm,
    tools=[write_email, schedule_meeting, check_calendar_availability,
           manage_memory_tool, search_memory_tool],
    prompt=create_prompt,
    store=store,
)

email_agent = (
    StateGraph(State)
    .add_node(triage_router)
    .add_node("response_agent", response_agent)
    .add_edge(START, "triage_router")
    .compile(store=store)
)

CONFIG = {"configurable": {"langgraph_user_id": USER_ID}}


def run_agent(email_input):
    response = email_agent.invoke({"email_input": email_input}, config=CONFIG)
    for message in response.get("messages", []):
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            for call in tool_calls:
                print(f"  tool: {call['name']}({json.dumps(call['args'])[:90]})")
        elif getattr(message, "content", None) and message.__class__.__name__ == "AIMessage":
            print(f"  reply: {str(message.content)[:150]}")
    return response

In [ ]:
response = run_agent(SAMPLE_EMAIL)

## 6. Procedural Memory

The part that makes the assistant improve. `create_multi_prompt_optimizer` takes a conversation plus one line of plain-English feedback, decides **which** stored prompt the feedback is about, and rewrites it.

Each prompt carries a `when_to_update` note — that is the only signal the optimizer has for routing feedback to the right place, and section 7 shows how well a 7B does at it.

> The course notebook writes only `main_agent` back to the store and leaves the rest as an exercise. All four are handled here — otherwise feedback about triage gets computed and then silently discarded.

In [ ]:
def optimize_prompts(response, feedback: str):
    """Rewrite stored instructions from one piece of user feedback."""
    optimizer = create_multi_prompt_optimizer(llm, kind="prompt_memory")

    keys = {
        "main_agent": "agent_instructions",
        "triage-ignore": "triage_ignore",
        "triage-notify": "triage_notify",
        "triage-respond": "triage_respond",
    }
    when_to_update = {
        "main_agent": "Update whenever there is feedback on how the agent should write emails or schedule events",
        "triage-ignore": "Update whenever there is feedback on which emails should be ignored",
        "triage-notify": "Update whenever there is feedback on which emails the user should be notified of",
        "triage-respond": "Update whenever there is feedback on which emails deserve a response",
    }
    prompts = [
        {
            "name": name,
            "prompt": store.get((USER_ID,), key).value["prompt"],
            "update_instructions": "keep the instructions short and to the point",
            "when_to_update": when_to_update[name],
        }
        for name, key in keys.items()
    ]

    try:
        updated = optimizer.invoke({"trajectories": [(response["messages"], feedback)], "prompts": prompts})
    except IndexError:
        # langmem reads result["responses"][0].which to decide what to rewrite.
        # A small model that fails to emit that choice leaves the list empty and
        # langmem raises rather than degrading. Roughly 1 run in 3 on qwen2.5:7b.
        print("  optimizer returned no prompt selection this run (small-model failure)")
        return []

    changed = []
    for new, old in zip(updated, prompts):
        if new["prompt"] != old["prompt"]:
            store.put((USER_ID,), keys[old["name"]], {"prompt": clean_prompt(new["prompt"])})
            changed.append(old["name"])
    return changed

In [ ]:
print("before:", store.get((USER_ID,), "agent_instructions").value["prompt"])

changed = optimize_prompts(response, "Always sign your emails `John Doe`")
print("\nrewritten:", changed)
print("\nafter: ", store.get((USER_ID,), "agent_instructions").value["prompt"])

Feedback about *how to write emails* correctly moved `main_agent` and left the three triage rules untouched — which is the behaviour you want, and the reason `when_to_update` exists.

**Why `clean_prompt` is needed.** LangMem wraps the prompt being optimised in `<current_prompt>` tags. A 7B model occasionally copies the opening tag into its rewrite, which then gets stored and fed back on the next turn, compounding. Larger models do not do this; stripping the tags is cheap insurance.

## 7. Watching Behaviour Change

Now feedback aimed at *triage* rather than at writing. Ideally `triage_ignore` absorbs it and the same email stops being answered.

In [ ]:
nuisance = {
    "author": "Alice Jones <alice.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": "Hi John,\n\nUrgent issue - your service is down. Is there a reason why?",
}

print("Before feedback:")
response = run_agent(nuisance)
print("\ntriage_ignore before:", store.get((USER_ID,), "triage_ignore").value["prompt"])

In [ ]:
changed = optimize_prompts(response, "Ignore any emails from Alice Jones")
print("rewritten:", changed)
print("triage_ignore after:", store.get((USER_ID,), "triage_ignore").value["prompt"])

print("\nAfter feedback (same email again):")
run_agent(nuisance)

### The honest limit of running this loop on a 7B

Deciding **which** prompt a piece of feedback belongs to is the hardest step in the whole notebook, and `qwen2.5:7b` is not reliable at it. Running this exact feedback three times gave three different answers — `triage-respond`, `triage-ignore`, `main_agent` — **despite `temperature=0`**, because the optimizer makes several internal calls and only one of them has to wobble.

There is a third outcome: LangMem decides what to rewrite by reading `result["responses"][0].which`, and when the model fails to emit that structured choice the list is empty and LangMem raises `IndexError` rather than degrading. `optimize_prompts` catches it and reports "no prompt selection" — without that guard the notebook simply dies here, roughly one run in three.

So expect the cell above to sometimes rewrite the wrong prompt, and sometimes rewrite nothing at all. Both are real results, not a broken notebook.

The practical reading:

| Component | Local 7B verdict |
|---|---|
| Triage with structured output | Works well |
| ReAct agent calling 5 tools | Works well |
| Semantic + episodic retrieval | Works well (embeddings are local and deterministic) |
| **Prompt optimizer** | **Unreliable — wrong prompt, or an `IndexError`. Point this one at a stronger model first** |

The optimizer is a single argument (`create_multi_prompt_optimizer(llm, ...)`), so upgrading just that piece while the agent stays local is a one-line change.

## Summary

- **Memory is three mechanisms, not one.** Semantic makes the agent informed, episodic makes it consistent, procedural makes it *change*.
- **One store, namespaced per user** holds all three; swapping in a persistent store never touches agent code.
- **Read prompts from the store per invocation.** Closing over a constant is what would make the agent unable to learn.
- **`when_to_update` is the routing signal** that decides which prompt absorbs a piece of feedback — and the weakest link on a small model.
- **Structured output does the triage**, so routing is a value to branch on rather than prose to parse.

### Where to go next

- Swap `InMemoryStore` for a persistent store so memory survives restart.
- Point `create_multi_prompt_optimizer` at a stronger model and rerun section 7 a few times.
- Feed the agent's tool-calling contract through the [Pydantic tutorial](../../pydantic/notebooks/pydantic-tutorial.ipynb) patterns.
- Score the assistant's replies with [DeepEval](../../../evaluation/deepeval/notebooks/deepeval-evaluation-tutorial.ipynb).

**Adapted from** DeepLearning.AI's *Long-Term Agentic Memory with LangGraph*, lesson 5 — moved off `gpt-4o`/`claude-3-5-sonnet` onto a local model, with the missing `prompts` module inlined and the store-writing exercise completed.